# Lab 06 — 06 Volume-Drop Alert Simulation

The Synthea sample is a **static historical dataset**, so this notebook creates
a controlled low-volume test month instead of pretending that a live feed exists.

The Gold facts are **not modified**.

## Flow

```text
fact_encounters
      ↓
real monthly counts
      ↓
baseline = average of 3 months before the latest source month
      ↓
simulated next month = 20% of baseline
      ↓
volume-drop metric
      ↓
SQL Alert evaluates should_alert = 1
```

Default alert threshold: **30% drop**.

## 1. Runtime parameters

In [0]:
dbutils.widgets.text("catalog", "dbr_dev", "01 Catalog")
dbutils.widgets.text("schema", "parvinbadalov", "02 Schema")
dbutils.widgets.text("volume_name", "lab06_gold_analytics", "03 External volume")
dbutils.widgets.text("drop_threshold_pct", "30", "04 Drop threshold %")
dbutils.widgets.text("simulated_volume_pct", "20", "05 Simulated volume %")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume_name = dbutils.widgets.get("volume_name")
drop_threshold_pct = float(dbutils.widgets.get("drop_threshold_pct"))
simulated_volume_pct = float(dbutils.widgets.get("simulated_volume_pct"))

if not (0 <= drop_threshold_pct <= 100):
    raise ValueError("drop_threshold_pct must be between 0 and 100.")

if not (0 <= simulated_volume_pct <= 100):
    raise ValueError("simulated_volume_pct must be between 0 and 100.")

print(f"Drop threshold       : {drop_threshold_pct:.1f}%")
print(f"Simulated volume     : {simulated_volume_pct:.1f}% of baseline")

## 2. Shared configuration

In [0]:
import sys
from pathlib import Path
from pyspark.sql import functions as F

current_dir = Path.cwd()
lab_root = current_dir.parent if current_dir.name == "notebooks" else current_dir

if str(lab_root) not in sys.path:
    sys.path.insert(0, str(lab_root))

from src.config import Lab06Config

config = Lab06Config(
    catalog=catalog,
    schema=schema,
    volume_name=volume_name,
)

metrics_table = config.table("lab06_data_volume_metrics")

print(f"Source fact   : {config.fact_encounters}")
print(f"Metrics table : {metrics_table}")

## 3. Build the real monthly encounter profile

In [0]:
monthly_counts = (
    spark.table(config.fact_encounters)
    .filter(F.col("encounter_date").isNotNull())
    .groupBy(
        F.trunc("encounter_date", "month").alias("encounter_month")
    )
    .agg(
        F.count("*").alias("encounter_count")
    )
    .orderBy("encounter_month")
)

print(f"Observed months: {monthly_counts.count():,}")

display(
    monthly_counts
    .orderBy(F.desc("encounter_month"))
    .limit(12)
    .orderBy("encounter_month")
)

## 4. Select baseline months

To avoid treating the final source month as a complete production month, the
baseline uses the **three observed months immediately before the latest source month**.

In [0]:
month_rows = (
    monthly_counts
    .orderBy(F.desc("encounter_month"))
    .limit(4)
    .collect()
)

if len(month_rows) < 4:
    raise RuntimeError(
        "At least four observed months are required for the alert simulation."
    )

latest_source_month = month_rows[0]["encounter_month"]
baseline_rows = month_rows[1:4]

baseline_months = sorted(
    [row["encounter_month"] for row in baseline_rows]
)

baseline_counts = [
    int(row["encounter_count"])
    for row in baseline_rows
]

baseline_count = int(round(sum(baseline_counts) / len(baseline_counts)))

if baseline_count <= 0:
    raise RuntimeError("Baseline encounter count must be greater than zero.")

print(f"Latest source month : {latest_source_month}")
print(f"Baseline months     : {baseline_months[0]} -> {baseline_months[-1]}")
print(f"Baseline counts     : {sorted(baseline_counts)}")
print(f"Baseline average    : {baseline_count:,}")

## 5. Create a controlled low-volume month

In [0]:
test_month = (
    spark.createDataFrame([(latest_source_month,)], ["latest_month"])
    .select(
        F.add_months("latest_month", 1).alias("test_month")
    )
    .first()["test_month"]
)

simulated_count = int(
    round(
        baseline_count
        * simulated_volume_pct
        / 100.0
    )
)

volume_drop_pct = round(
    (
        (baseline_count - simulated_count)
        / baseline_count
        * 100.0
    ),
    2,
)

should_alert = int(
    volume_drop_pct >= drop_threshold_pct
)

alert_status = (
    "TRIGGERED"
    if should_alert == 1
    else "OK"
)

print(f"Test month       : {test_month}")
print(f"Baseline count   : {baseline_count:,}")
print(f"Simulated count  : {simulated_count:,}")
print(f"Volume drop      : {volume_drop_pct:.2f}%")
print(f"Threshold        : {drop_threshold_pct:.2f}%")
print(f"Should alert     : {should_alert}")
print(f"Status           : {alert_status}")

## 6. Persist the alert metric

This table is deliberately separate from the Gold fact tables. Re-running the
notebook replaces only the simulation metric and never deletes encounter data.

In [0]:
metric_df = (
    spark.createDataFrame(
        [(
            "fact_encounters",
            baseline_months[0],
            baseline_months[-1],
            test_month,
            baseline_count,
            simulated_count,
            float(volume_drop_pct),
            float(drop_threshold_pct),
            should_alert,
            alert_status,
        )],
        [
            "data_source",
            "baseline_start_month",
            "baseline_end_month",
            "test_month",
            "baseline_encounter_count",
            "observed_encounter_count",
            "volume_drop_pct",
            "drop_threshold_pct",
            "should_alert",
            "alert_status",
        ],
    )
    .withColumn(
        "generated_at",
        F.current_timestamp(),
    )
)

(
    metric_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(metrics_table)
)

display(spark.table(metrics_table))

## 7. Validate the simulated alert condition

In [0]:
result = spark.table(metrics_table).first()

expected_drop_pct = round(
    (
        (
            result["baseline_encounter_count"]
            - result["observed_encounter_count"]
        )
        / result["baseline_encounter_count"]
        * 100.0
    ),
    2,
)

checks = [
    (
        "baseline_positive",
        result["baseline_encounter_count"] > 0,
    ),
    (
        "simulated_volume_below_baseline",
        result["observed_encounter_count"]
        < result["baseline_encounter_count"],
    ),
    (
        "drop_percentage_reconciles",
        float(result["volume_drop_pct"])
        == float(expected_drop_pct),
    ),
    (
        "alert_triggered",
        int(result["should_alert"]) == 1,
    ),
]

validation_df = spark.createDataFrame(
    [
        (
            check_name,
            "PASS" if passed else "FAIL",
        )
        for check_name, passed in checks
    ],
    ["check_name", "status"],
)

display(validation_df)

failed_checks = [
    check_name
    for check_name, passed in checks
    if not passed
]

if failed_checks:
    raise RuntimeError(
        "Alert simulation validation failed: "
        + ", ".join(failed_checks)
    )

## 8. Preview the SQL Alert result

The actual Databricks SQL Alert will evaluate `should_alert`.

The query always returns one row, which makes the alert state explicit instead
of relying on empty-query-result behavior.

In [0]:
alert_result_df = spark.sql(f'''
SELECT
    should_alert,
    alert_status,
    data_source,
    test_month,
    baseline_encounter_count,
    observed_encounter_count,
    volume_drop_pct,
    drop_threshold_pct,
    generated_at
FROM {metrics_table}
LIMIT 1
''')

display(alert_result_df)

## 9. Completion

Expected result with the default parameters:

```text
simulated_volume_pct = 20
drop_threshold_pct   = 30

volume drop          ≈ 80%
should_alert         = 1
alert_status         = TRIGGERED
```

**Next:** create a Databricks SQL Alert v2 with:

- source column: `should_alert`
- comparison: `EQUAL`
- threshold: `1`
- notification: user email
- query: `sql/alert_volume_drop.sql`

In [0]:
print("LAB 06 — VOLUME-DROP SIMULATION COMPLETE")
print(f"Metrics table : {metrics_table}")
print(f"Alert status  : {alert_status}")
print(f"Should alert  : {should_alert}")
print("Next: configure SQL Alert v2 + email notification")